# Explicabilité — Faster R-CNN (ResNet50 FPN v2) sur FracAtlas — **5 seeds**
**Volet article** : jeu de test complet, analyses A1/A2 (tests d'intervention) et carte d'occlusion (exploratoire), pour **les 5 seeds d'entraînement** (42, 123, 2024, 7, 77). Protocole : *volet explicabilité, v1.1* ; l'extension aux 5 seeds était reportée au §7.1.

### Règles appliquées
- **R1** perturbations sur l'image 640×640 prétraitée, avant les normalisations · **R2/R7** seuil de score 0,001, 300 détections, fp32, mode évaluation, meilleur checkpoint · **R3** τ (Youden) calculé sur la validation, **pour chaque seed** · **R4** mêmes images pour tous les seeds (échantillon tiré une fois, graine 2026) · **R10** interventions dans la région de contenu (hors gris du letterbox).
- Le jeu de test est verrouillé tant que `PARAMS_FROZEN = False`.

### Fonctionnement
- Les cellules 1 à 6 se lancent **une fois** (configuration, extraction, prétraitement, fonctions, échantillon). La **cellule 9** boucle sur les 5 seeds et enregistre tout dans `explicabilite_5seeds/seed_<n>/`. La **cellule 10** agrège les 5 seeds.
- **Reprise automatique** : chaque analyse enregistre son résultat ; après une déconnexion, relancer les cellules 1 à 9 saute ce qui est terminé. `FORCE_RECOMPUTE = True` pour tout refaire.
- **Durée indicative** (T4) : ≈ 35 à 40 min par seed, soit **3 h à 3 h 30** pour 5 seeds. Pour raccourcir : `RUN_OCCLUSION = False`.
- **Test rapide d'abord** : `DEBUG_N = 15` (2 seeds seulement, sorties dans `…_debug`, ≈ 10 min) pour détecter une erreur avant le lancement complet.

### Interprétation entre seeds
Chaque seed donne un modèle différent évalué sur **les mêmes images** : l'IC95 % d'un seed mesure l'incertitude liée aux images ; l'écart entre seeds mesure la variabilité d'entraînement. Les résultats sont rapportés **par seed**, puis en moyenne ± écart-type (ddof = 1), avec le nombre de seeds où chaque hypothèse est soutenue.

### Héritage du run 2 (à déclarer dans l'article)
Correctif R10 du flou (calculé sur la région de contenu) ; ajouts **exploratoires** : survie au niveau image, indice sur les vrais positifs, carte d'occlusion.

## Cellule 1 — Configuration

In [1]:
import os, shutil
from google.colab import drive
# Drive monté AVANT toute lecture ou création de dossier (sinon un faux dossier local /content/drive peut bloquer le montage)
if os.path.ismount('/content/drive'):
    print('Drive déjà monté.')
else:
    shutil.rmtree('/content/drive', ignore_errors=True)   # faux dossier local uniquement
    drive.mount('/content/drive')

import os, json, random, math, hashlib, datetime, copy, gc, time
import numpy as np
import pandas as pd

# ─── MODES D'EXÉCUTION ──────────────────────────────────────────────────────────
DEBUG_N        = None     # ex. 15 : TEST RAPIDE (15 fracturées + 15 non fracturées, 2 seeds). None = exécution complète.
SEEDS_RUN      = [42, 123, 2024, 7, 77]   # les 5 seeds d'entraînement (identiques à l'entraînement des modèles)
RUN_A1A2       = True
RUN_OCCLUSION  = True     # carte d'occlusion : EXPLORATOIRE (non prévue au protocole v1.1)
OCC_ALL_TP     = False    # False : vrais positifs de l'échantillon ; True : tous les vrais positifs du split (long)
OCC_WIN, OCC_STRIDE, OCC_FILL, OCC_BATCH = 80, 48, 'median_ring', 8
FORCE_RECOMPUTE = False   # True : ignorer les résultats déjà enregistrés
RUN_TAG        = '5seeds' # sorties dans explicabilite_<RUN_TAG>/seed_<n>/ ; les runs 1 et 2 (seed 42) ne sont pas touchés

# ─── CHEMINS ────────────────────────────────────────────────────────────────────
FRACATLAS_ZIP  = '/content/drive/MyDrive/FracAtlas/archive.zip'
FRACATLAS_ROOT = '/content/FracAtlas_extracted'
SPLITS_PATH    = '/content/drive/MyDrive/resultats_finaux_fracture/splits_fracatlas.json'
STATS_PATH     = '/content/drive/MyDrive/resultats_finaux_fracture/fracatlas_clahe_stats.json'
_CANDIDATES    = ['/content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN',
                  '/content/drive/MyDrive/resultats_finaux_fracture/FasterRCNN']
SAVE_DIR = next((p for p in _CANDIDATES if os.path.exists(os.path.join(p, 'seed_42', 'best.pth'))), None)
assert SAVE_DIR, f'Dossier des checkpoints Faster R-CNN introuvable parmi : {_CANDIDATES}'
for _s in SEEDS_RUN:
    assert os.path.exists(os.path.join(SAVE_DIR, f'seed_{_s}', 'best.pth')), f'Checkpoint manquant : seed {_s}'
XAI_DIR_MAIN   = os.path.join(SAVE_DIR, 'explicabilite')            # run 1 (seed 42)
LEGACY_DIRS    = [XAI_DIR_MAIN, XAI_DIR_MAIN + '_run2']             # runs précédents : τ, scores de base, échantillon relus s'ils existent
XAI_DIR        = XAI_DIR_MAIN + f'_{RUN_TAG}' + ('_debug' if DEBUG_N else '')
PREP_DIR       = '/content/xai_frcnn_dataset'      # prétraitement local (non persistant entre sessions)
os.makedirs(XAI_DIR, exist_ok=True)
print('Checkpoints :', SAVE_DIR, '\nSorties     :', XAI_DIR)
if DEBUG_N: print(f'⚠️ MODE TEST RAPIDE (DEBUG_N = {DEBUG_N}) : ne pas utiliser ces résultats dans l\'article.')

# ─── PRÉTRAITEMENT (identique à l'évaluation de référence) ──────────────────────
IMG_SIZE   = 640
CLAHE_CLIP = 2.0
CLAHE_GRID = (8, 8)

# ─── MODÈLE ─────────────────────────────────────────────────────────────────────
SCORE_THRESH = 0.001       # R2
MAX_DETS     = 300         # R2
IOU_MATCH    = 0.5         # appariement lésion (R2)

# ─── PÉRIMÈTRE : pilote (validation) ou analyse (test) ──────────────────────────
SPLIT          = 'test'    # 'val' = pilote ; 'test' = analyse finale pour l'article
PARAMS_FROZEN  = True      # paramètres figés (v1.2) : identiques à ceux du pilote et des runs 1 et 2
PARAMS_VERSION = f'v1.2 — {datetime.date.today().isoformat()}'

# ─── PARAMÈTRES (protocole §6) — figés, ne plus modifier ────────────────────────
BOX_MARGIN      = 0.10     # marge autour des boîtes annotées (A1)
CTRL_PERCENTILE = 25       # témoin sur du tissu : intensité moyenne > ce centile (A2)
N_CTRL_DRAWS    = 5        # tirages du témoin par image (A2)
RING_PX         = 15       # largeur de l'anneau pour le remplissage « médiane locale »
BLUR_SIGMA      = 25       # flou gaussien « fort » (écart-type en pixels)
FILL_METHODS    = ['median_ring', 'blur']   # R8
SHOW_N          = 6        # figures affichées dans le notebook (premier seed seulement ; toutes sont enregistrées)

# ─── STATISTIQUES ───────────────────────────────────────────────────────────────
N_PERM  = 1000             # permutations (indicateur de la carte d'occlusion)
N_BOOT  = 1000             # bootstrap (par image)
SAMPLE_SEED_TEST  = 2026   # graine de l'échantillon d'analyse sur le test (protocole §5)
SAMPLE_SEED_PILOT = 0      # graine de l'échantillon pilote (validation)

def assert_split_allowed(split):
    if split == 'test' and not PARAMS_FROZEN:
        raise RuntimeError("Jeu de test verrouillé : les paramètres ne sont pas figés. Mets PARAMS_FROZEN = True et renseigne PARAMS_VERSION.")
    if split == 'test' and not PARAMS_VERSION:
        raise RuntimeError("Renseigne PARAMS_VERSION (version figée et datée) avant d'utiliser le test.")
assert_split_allowed(SPLIT)

with open(SPLITS_PATH) as f: splits = json.load(f)
with open(STATS_PATH) as f:  clahe_stats = json.load(f)
seeds_to_run = SEEDS_RUN[:2] if DEBUG_N else SEEDS_RUN
print(f"Split : {SPLIT} | seeds : {seeds_to_run} | paramètres figés : {PARAMS_FROZEN} ({PARAMS_VERSION})")
print({k: len(v) for k, v in splits.items()})

Mounted at /content/drive
Checkpoints : /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN 
Sorties     : /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN/explicabilite_5seeds
Split : test | seeds : [42, 123, 2024, 7, 77] | paramètres figés : True (v1.2 — 2026-11-01)
{'train': 2857, 'val': 613, 'test': 613}


## Cellule 2 — Archive FracAtlas : extraction contrôlée et index des fichiers

In [2]:
from google.colab import drive
import zipfile
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

import shutil
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
EXPECTED_IMAGES = 4083      # 717 fracturées + 3366 non fracturées (protocole §2.1)
def _count_images(root):
    if not os.path.isdir(root): return 0
    return sum(os.path.splitext(f)[1].lower() in IMG_EXTENSIONS for _, _, fs in os.walk(root) for f in fs)

n_img = _count_images(FRACATLAS_ROOT)
if n_img == EXPECTED_IMAGES:
    print(f'✅ Archive déjà extraite et complète ({n_img} images).')
else:
    if n_img: print(f'⚠️ Extraction incomplète ({n_img} images au lieu de {EXPECTED_IMAGES}) : on repart de zéro.')
    assert os.path.exists(FRACATLAS_ZIP), f'Archive introuvable : {FRACATLAS_ZIP}'
    shutil.rmtree(FRACATLAS_ROOT, ignore_errors=True); os.makedirs(FRACATLAS_ROOT, exist_ok=True)
    with zipfile.ZipFile(FRACATLAS_ZIP, 'r') as zf: zf.extractall(FRACATLAS_ROOT)
    n_img = _count_images(FRACATLAS_ROOT)
    assert n_img == EXPECTED_IMAGES, f'Après extraction : {n_img} images au lieu de {EXPECTED_IMAGES}'
    print(f'✅ Extraction terminée et vérifiée ({n_img} images).')
_image_index, _label_index = {}, {}
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    for fname in files:
        if os.path.splitext(fname)[1].lower() in IMG_EXTENSIONS:
            _image_index[fname] = os.path.join(root, fname)
        if fname.endswith('.txt') and 'Annotations' in root:
            _label_index[fname] = os.path.join(root, fname)
print(f'{len(_image_index)} images indexées | {len(_label_index)} fichiers de labels indexés (attendu ≈ 4083)')
find_image_path = lambda image_id: _image_index.get(image_id)
find_label_path = lambda stem: _label_index.get(stem + '.txt')

✅ Extraction terminée et vérifiée (4083 images).
4083 images indexées | 4084 fichiers de labels indexés (attendu ≈ 4083)


## Cellule 3 — Prétraitement (CLAHE + letterbox) et région de contenu
Fonctions identiques à celles des notebooks d'entraînement ; on enregistre en plus, pour chaque image, la région de contenu [x0, y0, x1, y1] (hors remplissage), nécessaire à la règle R10.

In [3]:
import cv2
from tqdm.auto import tqdm

def apply_clahe(img_gray):
    return cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID).apply(img_gray)

def letterbox_image(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_y, pad_x = (size - new_h) // 2, (size - new_w) // 2
    canvas = np.full((size, size), 114, dtype=np.uint8)
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y, new_w, new_h

def letterbox_yolo_boxes(label_path, orig_w, orig_h, scale, pad_x, pad_y, size=IMG_SIZE):
    if label_path is None or not os.path.exists(label_path):
        return []
    out = []
    with open(label_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) != 5: continue
            cls, xc, yc, bw, bh = p
            xc, yc, bw, bh = float(xc) * orig_w, float(yc) * orig_h, float(bw) * orig_w, float(bh) * orig_h
            out.append(f'{cls} {(xc*scale+pad_x)/size:.6f} {(yc*scale+pad_y)/size:.6f} {bw*scale/size:.6f} {bh*scale/size:.6f}')
    return out

def preprocess_split(image_ids, split_name):
    img_dir = os.path.join(PREP_DIR, 'images', split_name)
    lbl_dir = os.path.join(PREP_DIR, 'labels', split_name)
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)
    regions, n_missing = {}, 0
    for image_id in tqdm(image_ids, desc=f'Prétraitement {split_name}'):
        stem = os.path.splitext(image_id)[0]
        src = find_image_path(image_id)
        if src is None:
            n_missing += 1; continue
        img = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
        oh, ow = img.shape[:2]
        lb, scale, px, py, nw, nh = letterbox_image(apply_clahe(img), IMG_SIZE)
        cv2.imwrite(os.path.join(img_dir, stem + '.jpg'), cv2.cvtColor(lb, cv2.COLOR_GRAY2BGR))
        lines = letterbox_yolo_boxes(find_label_path(stem), ow, oh, scale, px, py, IMG_SIZE)
        with open(os.path.join(lbl_dir, stem + '.txt'), 'w') as f: f.write('\n'.join(lines))
        regions[stem] = [int(px), int(py), int(px + nw), int(py + nh)]
    assert n_missing == 0, f'{n_missing} images introuvables pour {split_name}'
    with open(os.path.join(XAI_DIR, f'content_regions_{split_name}.json'), 'w') as f: json.dump(regions, f)
    return regions

SPLITS_TO_PREPARE = ['val'] + (['test'] if PARAMS_FROZEN else [])    # R3 : τ toujours sur val
content_regions = {}
for sp in SPLITS_TO_PREPARE:
    content_regions[sp] = preprocess_split(splits[sp], sp)

if DEBUG_N:     # sous-ensemble équilibré, uniquement sur le split analysé (la validation reste complète pour τ)
    _lab = lambda sp, s: os.path.getsize(os.path.join(PREP_DIR, 'labels', sp, s + '.txt')) > 0
    _fr = [s for s in sorted(content_regions[SPLIT]) if _lab(SPLIT, s)]
    _ng = [s for s in sorted(content_regions[SPLIT]) if not _lab(SPLIT, s)]
    _keep = set(_fr[:DEBUG_N] + _ng[:DEBUG_N])
    content_regions[SPLIT] = {s: v for s, v in content_regions[SPLIT].items() if s in _keep}
    print(f'Mode test rapide : {len(content_regions[SPLIT])} images conservées pour {SPLIT}.')

n_frac = {sp: sum(os.path.getsize(os.path.join(PREP_DIR, 'labels', sp, s + '.txt')) > 0 for s in content_regions[sp])
          for sp in content_regions}
print('Images fracturées par split préparé :', n_frac, '(attendu : val = 96 ; test = 108)')
ex = next(iter(content_regions['val'].items()))
print('Exemple de région de contenu :', ex)

Prétraitement val:   0%|          | 0/613 [00:00<?, ?it/s]

Prétraitement test:   0%|          | 0/613 [00:00<?, ?it/s]

Images fracturées par split préparé : {'val': 96, 'test': 108} (attendu : val = 96 ; test = 108)
Exemple de région de contenu : ('IMG0003274', [57, 0, 583, 640])


## Cellule 4 — Primitives liées au modèle (PyTorch)
Seules `load_model`, `predict` et `predict_batch` dépendent de PyTorch. Configuration d'origine : résolution interne 800 px, normalisation du jeu puis ImageNet. Pour adapter à YOLOv11n ou RT-DETR-l, remplacer ces fonctions.

In [4]:
import torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import cv2
from tqdm.auto import tqdm
from scipy.stats import spearmanr, binomtest
from sklearn.metrics import roc_curve, roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = 2  # fond + fracture

def build_model():
    m = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES)
    return m.to(device)

def load_model(seed):
    ckpt = os.path.join(SAVE_DIR, f'seed_{seed}', 'best.pth')
    m = build_model()
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    m.roi_heads.score_thresh = SCORE_THRESH           # R2
    m.roi_heads.detections_per_img = MAX_DETS         # R2
    for p in m.parameters(): p.requires_grad_(False)
    return m, ckpt

def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(block), b''):
            h.update(chunk)
    return h.hexdigest()

clahe_mean = np.array(clahe_stats['mean'], dtype=np.float32)
clahe_std  = np.array(clahe_stats['std'],  dtype=np.float32)

def load_img(split, stem):
    img = cv2.imread(os.path.join(PREP_DIR, 'images', split, stem + '.jpg'))
    assert img is not None, f'image introuvable : {split}/{stem}'
    return img                                          # BGR uint8, 640×640

def load_gt(split, stem):
    p = os.path.join(PREP_DIR, 'labels', split, stem + '.txt')
    boxes = []
    if os.path.exists(p) and os.path.getsize(p) > 0:
        for line in open(p):
            parts = line.strip().split()
            if len(parts) != 5: continue
            _, xc, yc, bw, bh = parts
            xc, yc, bw, bh = [float(v) * IMG_SIZE for v in (xc, yc, bw, bh)]
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)

def to_input(img_bgr):
    # Pipeline identique au Dataset : RGB, /255, normalisation CLAHE ; la normalisation ImageNet et le
    # redimensionnement à 800 px sont appliqués ensuite par le modèle (configuration d'origine).
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - clahe_mean) / clahe_std
    return torch.from_numpy(rgb.transpose(2, 0, 1)).float()

@torch.no_grad()
def predict(img_bgr, mdl=None):
    out = (mdl or model)([to_input(img_bgr).to(device)])[0]
    return out['boxes'].cpu().numpy(), out['scores'].cpu().numpy()

@torch.no_grad()
def predict_batch(imgs_bgr, mdl=None, bs=None):
    m = mdl or model; bs = bs or OCC_BATCH; out = []
    for i in range(0, len(imgs_bgr), bs):
        res = m([to_input(im).to(device) for im in imgs_bgr[i:i + bs]])
        out += [(r['boxes'].cpu().numpy(), r['scores'].cpu().numpy()) for r in res]
    return out

# Contrôle : les 3 canaux des images prétraitées sont-ils identiques ?
_s = sorted(content_regions['val'])[:20]
print('Écart max entre canaux (20 images) :', max(int(np.abs(load_img('val', s).astype(int)[:, :, 0] - load_img('val', s).astype(int)[:, :, 2]).max()) for s in _s))

Écart max entre canaux (20 images) : 0


## Cellule 5 — Fonctions de calcul (sans PyTorch)
Région de contenu, remplissages (R10 : le flou est calculé sur la région de contenu seule), témoins A2, scores, statistiques, carte d'occlusion.

In [5]:
def box_iou_np(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4); b = np.asarray(b, dtype=float).reshape(-1, 4)
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)

def scores_img_les(boxes, scores, gt):
    s_img = float(scores.max()) if len(scores) else 0.0
    s_les = 0.0
    if len(scores) and len(gt):
        ok = (box_iou_np(boxes, gt) >= IOU_MATCH).any(axis=1)
        if ok.any(): s_les = float(np.asarray(scores)[ok].max())
    return s_img, s_les

def region_mask(cr):
    m = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
    m[cr[1]:cr[3], cr[0]:cr[2]] = True
    return m

# ── Indicateur énergie dans la boîte / hors boîte (utilisé par la carte d'occlusion) ──
def boxes_mask(boxes, cmask):
    m = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
    for x1, y1, x2, y2 in boxes:
        m[int(max(0, round(y1))):int(min(IMG_SIZE, round(y2))), int(max(0, round(x1))):int(min(IMG_SIZE, round(x2)))] = True
    return m & cmask

def energy_ratio(cam, inside_mask, cmask):
    out_mask = cmask & ~inside_mask
    if inside_mask.sum() == 0 or out_mask.sum() == 0: return np.nan
    return float(cam[inside_mask].mean() / (cam[out_mask].mean() + 1e-9))

def random_boxes_like(gt, cr, rng):
    x0, y0, x1, y1 = cr
    out = []
    for g in gt:
        w, h = min(g[2] - g[0], x1 - x0), min(g[3] - g[1], y1 - y0)
        nx = rng.uniform(x0, x1 - w); ny = rng.uniform(y0, y1 - h)
        out.append([nx, ny, nx + w, ny + h])
    return out

def permutation_indicator(cam, gt, cr, n_perm, rng):
    cmask = region_mask(cr)
    obs = energy_ratio(cam, boxes_mask(gt, cmask), cmask)
    ref = np.array([energy_ratio(cam, boxes_mask(random_boxes_like(gt, cr, rng), cmask), cmask) for _ in range(n_perm)])
    ref = ref[~np.isnan(ref)]
    p = (1 + np.sum(ref >= obs)) / (1 + len(ref))
    return obs, float(np.mean(ref)), float(p)

# ── Interventions A1/A2 ──────────────────────────────────────────────────────────
def expand_clip(box, margin, cr):
    w, h = box[2] - box[0], box[3] - box[1]
    x1 = max(cr[0], int(math.floor(box[0] - margin * w))); y1 = max(cr[1], int(math.floor(box[1] - margin * h)))
    x2 = min(cr[2], int(math.ceil(box[2] + margin * w)));  y2 = min(cr[3], int(math.ceil(box[3] + margin * h)))
    return [x1, y1, x2, y2]

def zone_mask(zones):
    m = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
    for x1, y1, x2, y2 in zones: m[y1:y2, x1:x2] = True
    return m

def fill_channel(ch, mask, cmask, method):
    out = ch.copy()
    if method == 'median_ring':
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * RING_PX + 1, 2 * RING_PX + 1))
        ring = cv2.dilate(mask.astype(np.uint8), k).astype(bool) & ~mask & cmask
        out[mask] = np.median(ch[ring]) if ring.any() else np.median(ch[cmask])
    elif method == 'blur':
        # R10 : le flou est calculé sur la région de contenu seule (le gris du letterbox ne doit pas déteindre)
        ys, xs = np.where(cmask); y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        blurred = ch.copy()
        blurred[y0:y1, x0:x1] = cv2.GaussianBlur(ch[y0:y1, x0:x1], (0, 0), BLUR_SIGMA, borderType=cv2.BORDER_REFLECT)
        out[mask] = blurred[mask]
    else:
        raise ValueError(method)
    return out

def apply_fill(img_bgr, mask, cmask, method):
    return np.stack([fill_channel(img_bgr[:, :, c], mask, cmask, method) for c in range(3)], axis=-1)

def intersects(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

def place_controls(zones, cr, gray, cmask, rng, tries=300):
    thr = np.percentile(gray[cmask], CTRL_PERCENTILE)
    placed = []
    for (x1, y1, x2, y2) in zones:
        w, h = x2 - x1, y2 - y1
        if w >= cr[2] - cr[0] or h >= cr[3] - cr[1]: return None
        for _ in range(tries):
            nx = int(rng.integers(cr[0], cr[2] - w + 1)); ny = int(rng.integers(cr[1], cr[3] - h + 1))
            cand = [nx, ny, nx + w, ny + h]
            if any(intersects(cand, z) for z in zones) or any(intersects(cand, p) for p in placed): continue
            if gray[ny:ny + h, nx:nx + w].mean() <= thr: continue
            placed.append(cand); break
        else:
            return None
    return placed

def boot_ci(x, n=None, seed=0):
    n = n or N_BOOT
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    r = np.random.default_rng(seed); m = [r.choice(x, len(x)).mean() for _ in range(n)]
    return float(np.mean(x)), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5)), int(len(x))

# ── Carte d'occlusion (exploratoire) ─────────────────────────────────────────────
def occlusion_windows(cr, win, stride):
    x0, y0, x1, y1 = cr
    wx, wy = min(win, x1 - x0), min(win, y1 - y0)
    xs = list(range(x0, x1 - wx + 1, stride)); ys = list(range(y0, y1 - wy + 1, stride))
    if xs[-1] != x1 - wx: xs.append(x1 - wx)
    if ys[-1] != y1 - wy: ys.append(y1 - wy)
    return [(x, y, x + wx, y + wy) for y in ys for x in xs]

def occlusion_map(img, cr, gt, base_s_les, predict_fn, score_fn, win, stride, fill, fill_fn):
    cmask = region_mask(cr); wins = occlusion_windows(cr, win, stride); imgs = []
    for (x1, y1, x2, y2) in wins:
        m = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool); m[y1:y2, x1:x2] = True
        gch = fill_fn(img[:, :, 0], m, cmask, fill)            # canaux identiques : un seul calcul
        imgs.append(np.stack([gch, gch, gch], axis=-1))
    res = predict_fn(imgs)
    drops = np.zeros((IMG_SIZE, IMG_SIZE), np.float32); cover = np.zeros_like(drops); wd = []
    for (x1, y1, x2, y2), (b, s) in zip(wins, res):
        d = base_s_les - score_fn(b, s, gt)[1]
        drops[y1:y2, x1:x2] += d; cover[y1:y2, x1:x2] += 1; wd.append(d)
    omap = np.divide(drops, cover, out=np.zeros_like(drops), where=cover > 0)
    return omap, wins, np.array(wd)

def occlusion_indicators(omap, wins, wd, gt, cr, n_perm, rng):
    cmask = region_mask(cr); pos = np.clip(omap, 0, None)
    inside = boxes_mask(gt, cmask); tot = float(pos[cmask].sum())
    part_chute = float(pos[inside].sum() / tot) if tot > 0 else np.nan
    part_surface = float(inside.sum() / cmask.sum())
    obs, ref_mean, p = permutation_indicator(pos, gt, cr, n_perm, rng)
    zones = [expand_clip(g, BOX_MARGIN, cr) for g in gt]
    hors = [d for w, d in zip(wins, wd) if not any(intersects(w, z) for z in zones)]
    return {'part_chute_dans_boite': part_chute, 'part_surface_boite': part_surface,
            'enrichissement': part_chute / part_surface if part_surface > 0 and part_chute == part_chute else np.nan,
            'ratio_obs': obs, 'ratio_aleatoire_moyen': ref_mean, 'p_permutation': p,
            'max_chute_hors_lesion': float(max(hors)) if hors else np.nan, 'n_fenetres_hors_lesion': int(len(hors)),
            'n_fenetres': int(len(wins))}

def map_corr(a, b, cr=None, size=160):
    """Spearman entre deux cartes ; restreint à la région de contenu si cr est fourni (cartes 640×640)."""
    if cr is not None:
        a, b = a[cr[1]:cr[3], cr[0]:cr[2]], b[cr[1]:cr[3], cr[0]:cr[2]]
        a, b = cv2.resize(a.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA), cv2.resize(b.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
    r = spearmanr(np.asarray(a).ravel(), np.asarray(b).ravel())[0]
    return float(r) if r == r else float('nan')

## Cellule 6 — Métadonnées et échantillon d'analyse (communs aux seeds, R4)
Échantillon tiré avec une graine fixe **avant d'examiner une carte**, stratifié par région du corps (métadonnées FracAtlas, `dataset.csv`). Test : 24 fracturées (≥ 3 par région) + 12 non fracturées, graine 2026, tiré **une seule fois** puis rechargé depuis le disque (y compris depuis les runs précédents).

In [6]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def shared_cached(name):
    """Fichier commun aux seeds : relu depuis ce run, sinon depuis un run précédent, sinon chemin d'écriture de ce run."""
    for d in [XAI_DIR] + ([] if DEBUG_N else LEGACY_DIRS):
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    return os.path.join(XAI_DIR, name)

def find_metadata():
    for root, _, files in os.walk(FRACATLAS_ROOT):
        for f in files:
            if f.lower() == 'dataset.csv': return os.path.join(root, f)
    return None

meta, _mp = None, find_metadata()
if _mp:
    meta = pd.read_csv(_mp); meta.columns = [c.strip().lower() for c in meta.columns]
    print('Métadonnées :', _mp, '| colonnes :', list(meta.columns))
else:
    print('⚠️ dataset.csv introuvable : échantillon non stratifié (acceptable pour le pilote, pas pour le test).')

REGION_COLS = ['hand', 'leg', 'hip', 'shoulder', 'mixed']
_meta_stem = meta['image_id'].astype(str).str.replace(r'\.\w+$', '', regex=True) if (meta is not None and 'image_id' in meta.columns) else None
def region_of(stem):
    if _meta_stem is None: return 'inconnu'
    row = meta[_meta_stem == stem]
    if len(row) == 0: return 'inconnu'
    for c in REGION_COLS:
        if c in row.columns and int(row.iloc[0][c]) == 1: return c
    return 'inconnu'

def draw_sample(split, n_frac, n_neg, seed, min_per_region, path):
    if os.path.exists(path):
        print('Échantillon déjà tiré — rechargé :', path); return json.load(open(path))
    rng = np.random.default_rng(seed)
    stems = sorted(content_regions[split])
    frac = [s for s in stems if len(load_gt(split, s)) > 0]
    neg  = [s for s in stems if len(load_gt(split, s)) == 0]
    by_reg = {}
    for s in frac: by_reg.setdefault(region_of(s), []).append(s)
    chosen = []
    for reg, lst in by_reg.items():
        k = min(min_per_region, len(lst))
        chosen += list(rng.choice(lst, size=k, replace=False))
    rest = [s for s in frac if s not in chosen]
    need = max(0, n_frac - len(chosen))
    if need: chosen += list(rng.choice(rest, size=min(need, len(rest)), replace=False))
    sample = {'fractured': sorted(map(str, chosen[:max(n_frac, len(chosen))])),
              'non_fractured': sorted(map(str, rng.choice(neg, size=min(n_neg, len(neg)), replace=False))),
              'seed': seed, 'split': split, 'regions': {s: region_of(s) for s in chosen}}
    json.dump(sample, open(path, 'w'), indent=2)
    return sample

if SPLIT == 'val':
    sample = draw_sample('val', n_frac=8, n_neg=4, seed=SAMPLE_SEED_PILOT, min_per_region=1, path=shared_cached('sample_pilot_val.json'))
else:
    sample = draw_sample('test', n_frac=24, n_neg=12, seed=SAMPLE_SEED_TEST, min_per_region=3, path=shared_cached('sample_test_seed2026.json'))
sample_stems = sample['fractured'] + sample['non_fractured']
print(len(sample['fractured']), 'fracturées +', len(sample['non_fractured']), 'non fracturées')

Métadonnées : /content/FracAtlas_extracted/FracAtlas/dataset.csv | colonnes : ['image_id', 'hand', 'leg', 'hip', 'shoulder', 'mixed', 'hardware', 'multiscan', 'fractured', 'fracture_count', 'frontal', 'lateral', 'oblique']
Échantillon déjà tiré — rechargé : /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN/explicabilite_5seeds/sample_test_seed2026.json
24 fracturées + 12 non fracturées


## Cellule 7 — Activation d'une seed
Charge le checkpoint de la seed, fixe les seuils de sortie (R2), calcule (ou relit) le seuil τ de Youden **sur la validation** (R3) et les scores de base sur le split analysé. Enregistre la configuration et le hash SHA-256 du checkpoint (traçabilité). Les valeurs de τ et les scores de base déjà calculés lors des runs précédents (seed 42) sont relus, pas recalculés.

In [7]:
model = None
det_flags = {}          # seed -> série booléenne : lésion détectée (s_les >= τ) pour chaque image fracturée

def seed_cached(name, in_debug=False):
    """Fichier propre à une seed : relu depuis ce run, sinon depuis un run précédent (même nom de fichier)."""
    for d in [SEED_DIR] + ([] if (DEBUG_N and not in_debug) else LEGACY_DIRS):
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    return os.path.join(SEED_DIR, name)

def activate_seed(seed):
    global MODEL_SEED, CKPT, model, SEED_DIR, config_log, TAU, tau_info, base_df, base_idx, base_metrics, statuses_all, statuses
    MODEL_SEED = seed
    SEED_DIR = os.path.join(XAI_DIR, f'seed_{seed}'); os.makedirs(SEED_DIR, exist_ok=True)
    model = None; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model, CKPT = load_model(seed)
    config_log = {
        'date': datetime.datetime.now().isoformat(timespec='seconds'),
        'split': SPLIT, 'model_seed': seed, 'checkpoint': CKPT, 'checkpoint_sha256': sha256(CKPT),
        'torch': torch.__version__, 'torchvision': torchvision.__version__, 'device': device,
        'score_thresh': SCORE_THRESH, 'max_dets': MAX_DETS, 'iou_match': IOU_MATCH,
        'params_frozen': PARAMS_FROZEN, 'params_version': PARAMS_VERSION, 'run_tag': RUN_TAG,
        'params': dict(BOX_MARGIN=BOX_MARGIN, CTRL_PERCENTILE=CTRL_PERCENTILE, N_CTRL_DRAWS=N_CTRL_DRAWS, RING_PX=RING_PX,
                       BLUR_SIGMA=BLUR_SIGMA, FILL_METHODS=FILL_METHODS,
                       OCC_WIN=OCC_WIN, OCC_STRIDE=OCC_STRIDE, OCC_FILL=OCC_FILL, N_PERM=N_PERM, N_BOOT=N_BOOT),
    }
    json.dump(config_log, open(os.path.join(SEED_DIR, f'config_{SPLIT}.json'), 'w'), indent=2)

    # τ (Youden) sur la validation — R3
    tau_path = seed_cached(f'tau_val_seed{seed}.json', in_debug=True)
    if os.path.exists(tau_path):
        tau_info = json.load(open(tau_path)); print(f'[seed {seed}] τ chargé depuis le disque.')
    else:
        rows = []
        for stem in tqdm(sorted(content_regions['val']), desc=f'[seed {seed}] inférence validation', leave=False):
            gt = load_gt('val', stem); b, s = predict(load_img('val', stem))
            s_img, s_les = scores_img_les(b, s, gt)
            rows.append({'stem': stem, 'y': int(len(gt) > 0), 's_img': s_img, 's_les': s_les})
        val_df = pd.DataFrame(rows)
        fpr, tpr, thr = roc_curve(val_df['y'], val_df['s_img']); j = int(np.argmax(tpr - fpr))
        tau_info = {'tau': float(thr[j]), 'sens_val': float(tpr[j]), 'spec_val': float(1 - fpr[j]),
                    'auc_val': float(roc_auc_score(val_df['y'], val_df['s_img'])), 'n_val': int(len(val_df))}
        tau_path = os.path.join(SEED_DIR, f'tau_val_seed{seed}.json')
        json.dump(tau_info, open(tau_path, 'w'), indent=2)
        val_df.to_csv(os.path.join(SEED_DIR, f'val_scores_seed{seed}.csv'), index=False)
    TAU = tau_info['tau']

    # scores de base sur le split analysé (avant toute perturbation)
    base_path = seed_cached(f'scores_{SPLIT}_seed{seed}.csv')
    if os.path.exists(base_path):
        base_df = pd.read_csv(base_path, dtype={'stem': str}); print(f'[seed {seed}] scores de base chargés depuis le disque.')
    else:
        rows = []
        for stem in tqdm(sorted(content_regions[SPLIT]), desc=f'[seed {seed}] inférence {SPLIT}', leave=False):
            gt = load_gt(SPLIT, stem); b, s = predict(load_img(SPLIT, stem))
            s_img, s_les = scores_img_les(b, s, gt)
            rows.append({'stem': stem, 'y': int(len(gt) > 0), 's_img': s_img, 's_les': s_les})
        base_df = pd.DataFrame(rows)
        base_path = os.path.join(SEED_DIR, f'scores_{SPLIT}_seed{seed}.csv'); base_df.to_csv(base_path, index=False)
    base_df = base_df[base_df['stem'].isin(content_regions[SPLIT])].reset_index(drop=True)
    base_idx = base_df.set_index('stem')
    _y, _s = base_df['y'].values, base_df['s_img'].values
    base_metrics = {
        'n_images': int(len(base_df)), 'n_fracturees': int(_y.sum()), 'AUC': float(roc_auc_score(_y, _s)),
        'sensibilite_image_au_tau_val': float((_s[_y == 1] >= TAU).mean()),
        'specificite_image_au_tau_val': float((_s[_y == 0] < TAU).mean()),
        'sensibilite_lesion_au_tau_val': float((base_df.loc[base_df.y == 1, 's_les'] >= TAU).mean()),
        'n_fracturees_flaggees_hors_lesion': int(((_y == 1) & (_s >= TAU) & (base_df['s_les'].values < TAU)).sum()),
        'tau_val': TAU,
    }
    json.dump(base_metrics, open(os.path.join(SEED_DIR, f'base_metrics_{SPLIT}.json'), 'w'), indent=2)
    print(f"[seed {seed}] AUC = {base_metrics['AUC']:.4f} | τ = {TAU:.3f} | sens. lésion = {base_metrics['sensibilite_lesion_au_tau_val']:.3f}")
    def _status(stem):
        r = base_idx.loc[stem]
        if r['y'] == 1: return 'TP' if r['s_les'] >= TAU else 'FN'
        return 'FP' if r['s_img'] >= TAU else 'TN'
    statuses_all = {s: _status(s) for s in base_idx.index}
    statuses = {s: statuses_all[s] for s in sample_stems if s in statuses_all}
    det_flags[seed] = pd.Series({s: statuses_all[s] == 'TP' for s in base_idx.index if base_idx.loc[s, 'y'] == 1})

## Cellule 8 — Analyses (fonctions)
Chaque analyse enregistre ses sorties dans le dossier de la seed et **se saute si son résumé existe déjà** (sauf `FORCE_RECOMPUTE`). Les paramètres et définitions sont ceux du protocole v1.1 : A1/A2 (masquage de la lésion et témoin), occlusion (exploratoire).

In [8]:
def _cached_summary(name):
    p = os.path.join(SEED_DIR, name)
    if os.path.exists(p) and not FORCE_RECOMPUTE:
        print(f'[seed {MODEL_SEED}] {name} : déjà enregistré — rechargé.'); return json.load(open(p))
    return None

def _dump(obj, name):
    json.dump(obj, open(os.path.join(SEED_DIR, name), 'w'), indent=2, default=float)

# ───────── A1 / A2 ─────────
def analysis_a1a2():
    out_name = f'resume_interventions_{SPLIT}.json'
    cached = _cached_summary(out_name)
    if cached is not None: return cached
    rng_ctrl = np.random.default_rng(2026)          # mêmes témoins pour tous les seeds (R4)
    iv_rows = []
    frac_stems = [s for s in sorted(content_regions[SPLIT]) if len(load_gt(SPLIT, s)) > 0]
    for stem in tqdm(frac_stems, desc=f'[seed {MODEL_SEED}] A1/A2', leave=False):
        img = load_img(SPLIT, stem); gt = load_gt(SPLIT, stem); cr = content_regions[SPLIT][stem]
        cmask = region_mask(cr); gray = img[:, :, 0]
        b, s = predict(img); s_img0, s0 = scores_img_les(b, s, gt)
        zones = [expand_clip(gg, BOX_MARGIN, cr) for gg in gt]
        zones = [z for z in zones if z[2] > z[0] and z[3] > z[1]]
        for method in FILL_METHODS:
            b1, s1 = predict(apply_fill(img, zone_mask(zones), cmask, method)); s_img_a1, s_a1 = scores_img_les(b1, s1, gt)
            s_a2_list, s_img_a2_list = [], []
            for _ in range(N_CTRL_DRAWS):
                ctrl = place_controls(zones, cr, gray, cmask, rng_ctrl)
                if ctrl is None: continue
                b2, s2 = predict(apply_fill(img, zone_mask(ctrl), cmask, method)); s_img_a2, s_a2 = scores_img_les(b2, s2, gt)
                s_a2_list.append(s_a2); s_img_a2_list.append(s_img_a2)
            iv_rows.append({'stem': stem, 'region': region_of(stem), 'n_boites': len(gt), 'remplissage': method,
                            's_avant': s0, 's_A1': s_a1,
                            's_A2': float(np.mean(s_a2_list)) if s_a2_list else np.nan, 'n_temoins_ok': len(s_a2_list),
                            's_img_avant': s_img0, 's_img_A1': s_img_a1,
                            'surv_lesion_A2': float(np.mean([v >= TAU for v in s_a2_list])) if s_a2_list else np.nan,
                            'surv_img_A2': float(np.mean([v >= TAU for v in s_img_a2_list])) if s_img_a2_list else np.nan})
    iv = pd.DataFrame(iv_rows)
    iv['detecte_avant'] = iv['s_avant'] >= TAU
    iv['detecte_A1'] = iv['s_A1'] >= TAU
    iv['detecte_img_avant'] = iv['s_img_avant'] >= TAU
    iv['detecte_img_A1'] = iv['s_img_A1'] >= TAU
    iv['delta_A1'] = iv['s_avant'] - iv['s_A1']
    iv['delta_A2'] = iv['s_avant'] - iv['s_A2']
    iv['indice_specificite'] = iv['delta_A1'] - iv['delta_A2']
    iv.to_csv(os.path.join(SEED_DIR, f'interventions_{SPLIT}.csv'), index=False)
    summ = {}
    for method, d in iv.groupby('remplissage'):
        det = d[d['detecte_avant']]
        mean, lo, hi, n = boot_ci(d['indice_specificite'])
        summ[method] = {'n_images': int(len(d)), 'n_avec_temoin': n,
                        'delta_A1_moyen': float(d['delta_A1'].mean()), 'delta_A2_moyen': float(d['delta_A2'].mean()),
                        'indice_specificite': mean, 'IC95_bas': lo, 'IC95_haut': hi,
                        'taux_survie_A1': float(det['detecte_A1'].mean()) if len(det) else np.nan, 'n_detectees_avant': int(len(det)),
                        'taux_survie_lesion_A2': float(det['surv_lesion_A2'].mean()) if len(det) else np.nan,
                        'n_detectees_image_avant': int(d['detecte_img_avant'].sum()),
                        'taux_survie_image_A1': float(d.loc[d['detecte_img_avant'], 'detecte_img_A1'].mean()) if d['detecte_img_avant'].any() else np.nan,
                        'taux_survie_image_A2': float(d.loc[d['detecte_img_avant'], 'surv_img_A2'].mean()) if d['detecte_img_avant'].any() else np.nan}
        if len(det) > 1:
            m_tp, lo_tp, hi_tp, n_tp = boot_ci(det['indice_specificite'])
            summ[method].update({'indice_specificite_TP': m_tp, 'IC95_bas_TP': lo_tp, 'IC95_haut_TP': hi_tp, 'n_TP_avec_temoin': n_tp})
    by_region = {}
    for (method, reg), d in iv.groupby(['remplissage', 'region']):
        by_region[f'{method}|{reg}'] = {'n': int(len(d)), 'indice_specificite_moyen': float(d['indice_specificite'].mean()),
                                        'delta_A1_moyen': float(d['delta_A1'].mean()), 'delta_A2_moyen': float(d['delta_A2'].mean())}
    out = {'par_remplissage': summ, 'par_region': by_region,
           'n_images_sans_temoin': int((iv['n_temoins_ok'] == 0).sum() / max(1, len(FILL_METHODS)))}
    _dump(out, out_name)
    print(pd.DataFrame(summ).T.round(3))
    return out

# ───────── Carte d'occlusion (exploratoire) ─────────
def analysis_occlusion():
    out_name = f'resume_occlusion_{SPLIT}.json'
    cached = _cached_summary(out_name)
    if cached is not None: return cached
    if OCC_ALL_TP: occ_stems = [s for s in sorted(content_regions[SPLIT]) if statuses_all.get(s) == 'TP']
    else:          occ_stems = [s for s in sample['fractured'] if statuses_all.get(s) == 'TP']
    rng_occ = np.random.default_rng(7); occ_rows, occ_maps = [], {}
    for stem in tqdm(occ_stems, desc=f'[seed {MODEL_SEED}] occlusion', leave=False):
        img = load_img(SPLIT, stem); gt = load_gt(SPLIT, stem); cr = content_regions[SPLIT][stem]
        base_s_les = float(base_idx.loc[stem, 's_les'])
        omap, wins, wd = occlusion_map(img, cr, gt, base_s_les, predict_batch, scores_img_les, OCC_WIN, OCC_STRIDE, OCC_FILL, fill_channel)
        ind = occlusion_indicators(omap, wins, wd, gt, cr, N_PERM, rng_occ)
        occ_rows.append({'stem': stem, 'region': region_of(stem), 'n_boites': int(len(gt)), 's_les_base': base_s_les, **ind})
        occ_maps[stem] = omap.astype(np.float16)
    occ_df = pd.DataFrame(occ_rows)
    occ_df.to_csv(os.path.join(SEED_DIR, f'occlusion_{SPLIT}.csv'), index=False)
    np.savez_compressed(os.path.join(SEED_DIR, f'occlusion_cartes_{SPLIT}.npz'), **occ_maps)
    if len(occ_df) == 0:
        out = {'n_images_TP': 0}; _dump(out, out_name); return out
    d = occ_df.dropna(subset=['ratio_obs']); n = len(d); k = int((d['p_permutation'] < 0.05).sum())
    out = {'n_images_TP': int(len(occ_df)), 'fenetre_px': OCC_WIN, 'pas_px': OCC_STRIDE, 'remplissage': OCC_FILL,
           'part_chute_dans_boite_mediane': float(occ_df['part_chute_dans_boite'].median()),
           'part_surface_boite_mediane': float(occ_df['part_surface_boite'].median()),
           'enrichissement_median': float(occ_df['enrichissement'].median()),
           'ratio_obs_median': float(d['ratio_obs'].median()) if n else np.nan, 'part_p_inf_0_05': k / n if n else np.nan,
           'p_binomial_vs_5pct': float(binomtest(k, n, 0.05, alternative='greater').pvalue) if n else np.nan,
           'max_chute_hors_lesion_mediane': float(occ_df['max_chute_hors_lesion'].median()),
           'max_chute_hors_lesion_q25_q75': [float(occ_df['max_chute_hors_lesion'].quantile(.25)), float(occ_df['max_chute_hors_lesion'].quantile(.75))]}
    _dump(out, out_name)
    print(json.dumps(out, indent=2, default=float))
    return out

# ───────── Figures de l'échantillon ─────────
SHOW_FIGS = False
def show_case(split, stem, save_dir, show=True):
    img = load_img(split, stem); gt = load_gt(split, stem); b, s = predict(img)
    s_img, s_les = scores_img_les(b, s, gt); cr = content_regions[split][stem]
    st = ('TP' if s_les >= TAU else 'FN') if len(gt) else ('FP' if s_img >= TAU else 'TN')
    fig, ax = plt.subplots(1, 1, figsize=(5, 5)); ax = np.atleast_1d(ax)
    for a in ax:
        a.imshow(img[:, :, ::-1]); a.axis('off')
        a.add_patch(Rectangle((cr[0], cr[1]), cr[2]-cr[0], cr[3]-cr[1], fill=False, ls=':', ec='yellow', lw=1))
        for gg in gt: a.add_patch(Rectangle((gg[0], gg[1]), gg[2]-gg[0], gg[3]-gg[1], fill=False, ec='lime', lw=2))
    for bb, ss in zip(b, s):
        if ss >= TAU: ax[0].add_patch(Rectangle((bb[0], bb[1]), bb[2]-bb[0], bb[3]-bb[1], fill=False, ec='red', lw=2))
    ax[0].set_title(f'seed {MODEL_SEED} — {stem} — {st} (s_img={s_img:.2f}, τ={TAU:.2f})', fontsize=9)
    os.makedirs(save_dir, exist_ok=True)
    fig.savefig(os.path.join(save_dir, f'{split}_{stem}.png'), dpi=120, bbox_inches='tight')
    if show: plt.show()
    plt.close(fig)

def analysis_figures():
    fig_dir = os.path.join(SEED_DIR, f'figures_{SPLIT}')
    if os.path.isdir(fig_dir) and len(os.listdir(fig_dir)) >= len(sample_stems) and not FORCE_RECOMPUTE:
        print(f'[seed {MODEL_SEED}] figures déjà enregistrées.'); return {'dossier': fig_dir}
    for i, stem in enumerate(sample_stems):
        show_case(SPLIT, stem, fig_dir, show=(SHOW_FIGS and i < SHOW_N))
    return {'dossier': fig_dir}

## Cellule 9 — Exécution sur les 5 seeds
Boucle sur les seeds : activation, puis chaque analyse (reprise automatique). Après une déconnexion, relancer les cellules 1 à 9. Le résumé de chaque seed est enregistré dans `explicabilite_5seeds/seed_<n>/resume_article_faster_<split>_seed<n>.json`.

In [9]:
all_summaries = {}
t_start = time.time()
for k, seed in enumerate(seeds_to_run):
    print('\n' + '=' * 78 + f'\nSEED {seed}  ({k + 1}/{len(seeds_to_run)})  —  écoulé : {(time.time() - t_start) / 60:.1f} min\n' + '=' * 78)
    activate_seed(seed)
    SHOW_FIGS = (k == 0)
    S = {'modele': 'Faster R-CNN ResNet50-FPN v2', 'seed': seed, 'split': SPLIT, 'version_parametres': PARAMS_VERSION,
         'parametres': config_log['params'], 'checkpoint_sha256': config_log['checkpoint_sha256'], 'performance_base': base_metrics}
    if RUN_A1A2:                          S['A1_A2'] = analysis_a1a2()
    if RUN_OCCLUSION:                     S['occlusion_exploratoire'] = analysis_occlusion()
    S['figures'] = analysis_figures()
    _dump(S, f'resume_article_faster_{SPLIT}_seed{seed}.json')
    all_summaries[seed] = S
print(f'\n✅ Terminé : {len(all_summaries)} seeds en {(time.time() - t_start) / 60:.1f} min.')


SEED 42  (1/5)  —  écoulé : 0.0 min
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:00<00:00, 178MB/s]


[seed 42] τ chargé depuis le disque.
[seed 42] scores de base chargés depuis le disque.
[seed 42] AUC = 0.8579 | τ = 0.427 | sens. lésion = 0.519
[seed 42] resume_interventions_test.json : déjà enregistré — rechargé.
[seed 42] resume_occlusion_test.json : déjà enregistré — rechargé.
[seed 42] figures déjà enregistrées.

SEED 123  (2/5)  —  écoulé : 0.2 min
[seed 123] τ chargé depuis le disque.
[seed 123] scores de base chargés depuis le disque.
[seed 123] AUC = 0.8561 | τ = 0.625 | sens. lésion = 0.500
[seed 123] resume_interventions_test.json : déjà enregistré — rechargé.
[seed 123] resume_occlusion_test.json : déjà enregistré — rechargé.
[seed 123] figures déjà enregistrées.

SEED 2024  (3/5)  —  écoulé : 0.4 min
[seed 2024] τ chargé depuis le disque.
[seed 2024] scores de base chargés depuis le disque.
[seed 2024] AUC = 0.8281 | τ = 0.228 | sens. lésion = 0.556
[seed 2024] resume_interventions_test.json : déjà enregistré — rechargé.
[seed 2024] resume_occlusion_test.json : déjà enre

## Cellule 10 — Agrégation entre seeds
Tableau par seed et moyenne ± écart-type (ddof = 1), nombre de seeds où H1 est soutenue (IC95 % de l'indice de spécificité > 0) ; **stabilité entre seeds** : accord sur la détection des lésions et corrélation de Spearman des cartes d'occlusion d'une même image.

In [10]:
def _g(d, *path, default=np.nan):
    for p in path:
        if not isinstance(d, dict) or p not in d: return default
        d = d[p]
    return d

rows = []
for seed, S in all_summaries.items():
    r = {'seed': seed, 'AUC': _g(S, 'performance_base', 'AUC'),
         'sens_image': _g(S, 'performance_base', 'sensibilite_image_au_tau_val'),
         'spec_image': _g(S, 'performance_base', 'specificite_image_au_tau_val'),
         'sens_lesion': _g(S, 'performance_base', 'sensibilite_lesion_au_tau_val'),
         'n_flaggees_hors_lesion': _g(S, 'performance_base', 'n_fracturees_flaggees_hors_lesion')}
    for f in FILL_METHODS:
        a = _g(S, 'A1_A2', 'par_remplissage', f)
        r.update({f'indice_{f}': _g(a, 'indice_specificite'), f'IC_bas_{f}': _g(a, 'IC95_bas'), f'indice_TP_{f}': _g(a, 'indice_specificite_TP'),
                  f'surv_lesion_A1_{f}': _g(a, 'taux_survie_A1'), f'surv_lesion_A2_{f}': _g(a, 'taux_survie_lesion_A2'),
                  f'surv_image_A1_{f}': _g(a, 'taux_survie_image_A1'), f'surv_image_A2_{f}': _g(a, 'taux_survie_image_A2')})
    o = _g(S, 'occlusion_exploratoire')
    r.update({'occ_enrichissement_median': _g(o, 'enrichissement_median'), 'occ_max_chute_hors_lesion_med': _g(o, 'max_chute_hors_lesion_mediane')})
    rows.append(r)
tab = pd.DataFrame(rows).set_index('seed')
num = tab.select_dtypes(include=[np.number])
agg = pd.DataFrame({'moyenne': num.mean(), 'ecart_type_ddof1': num.std(ddof=1)}).T
print('═══ Résultats par seed ═══'); print(tab.round(3).T)
print('\n═══ Moyenne ± écart-type entre seeds ═══'); print(agg.round(3).T)

print('\n═══ H1 (A1/A2) : seeds où l\'IC95 % de l\'indice de spécificité exclut 0 par le bas ═══')
for f in FILL_METHODS:
    ok = int((tab[f'IC_bas_{f}'] > 0).sum()); print(f'  {f:12s} : {ok} / {len(tab)} seeds')

# ─── Stabilité entre seeds ───
stab = {}
frac_idx = [s for s in sorted(content_regions[SPLIT]) if base_idx.loc[s, 'y'] == 1]
if len(det_flags) >= 2:
    mat = pd.DataFrame({sd: det_flags[sd].reindex(frac_idx) for sd in det_flags})
    n_det = mat.sum(axis=1)
    stab['accord_detection_lesions'] = {'nb_images_detectees_par_k_seeds': {int(k): int(v) for k, v in n_det.value_counts().sort_index().items()},
                                        'detectees_par_tous': int((n_det == len(mat.columns)).sum()), 'detectees_par_aucun': int((n_det == 0).sum()),
                                        'n_images_fracturees': int(len(mat))}
    print('\n═══ Accord entre seeds : lésions détectées par k seeds ═══'); print(stab['accord_detection_lesions'])
def _pairs(seeds): return [(a, b) for i, a in enumerate(seeds) for b in seeds[i + 1:]]
sd_list = list(all_summaries)
if RUN_OCCLUSION and len(sd_list) >= 2:
    npo = {sd: np.load(os.path.join(XAI_DIR, f'seed_{sd}', f'occlusion_cartes_{SPLIT}.npz')) for sd in sd_list
           if os.path.exists(os.path.join(XAI_DIR, f'seed_{sd}', f'occlusion_cartes_{SPLIT}.npz'))}
    cors = []
    for a, b in _pairs(list(npo)):
        for s in set(npo[a].files) & set(npo[b].files):
            cors.append(map_corr(npo[a][s].astype(np.float32), npo[b][s].astype(np.float32), content_regions[SPLIT][s]))
    if cors: stab['occlusion_spearman_entre_seeds'] = {'moyenne': float(np.nanmean(cors)), 'mediane': float(np.nanmedian(cors)), 'n_paires_images': len(cors)}
    print('Stabilité des cartes d\'occlusion (Spearman entre seeds, même image TP) :', stab.get('occlusion_spearman_entre_seeds'))

tab.to_csv(os.path.join(XAI_DIR, f'resume_multiseed_{SPLIT}.csv'))
json.dump({'seeds': sd_list, 'par_seed': tab.reset_index().to_dict(orient='records'), 'moyenne_et_ecart_type': agg.to_dict(),
           'stabilite': stab, 'version_parametres': PARAMS_VERSION, 'run_tag': RUN_TAG},
          open(os.path.join(XAI_DIR, f'resume_multiseed_{SPLIT}.json'), 'w'), indent=2, default=float)
print('\n💾 Enregistré :', os.path.join(XAI_DIR, f'resume_multiseed_{SPLIT}.json'))

═══ Résultats par seed ═══
seed                             42      123     2024    7       77  
AUC                             0.858   0.856   0.828   0.865   0.852
sens_image                      0.648   0.657   0.648   0.667   0.750
spec_image                      0.877   0.919   0.885   0.919   0.853
sens_lesion                     0.519   0.500   0.556   0.528   0.519
n_flaggees_hors_lesion         14.000  17.000  10.000  15.000  25.000
indice_median_ring              0.478   0.491   0.503   0.496   0.473
IC_bas_median_ring              0.393   0.398   0.411   0.408   0.386
indice_TP_median_ring           0.865   0.934   0.887   0.893   0.871
surv_lesion_A1_median_ring      0.000   0.000   0.000   0.000   0.000
surv_lesion_A2_median_ring      0.975   0.974   0.980   0.965   0.982
surv_image_A1_median_ring       0.314   0.268   0.571   0.361   0.444
surv_image_A2_median_ring       0.971   0.980   0.983   0.975   0.980
indice_blur                     0.459   0.478   0.471   0.491  

## Notes
- Chaque seed est un modèle distinct évalué sur **les mêmes images** (R4) : l'IC95 % d'un seed reflète l'incertitude due aux images ; l'écart entre seeds reflète la variabilité d'entraînement. Rapporter les deux ; ne pas fusionner les 5 seeds en un seul échantillon.
- Les ajouts marqués *exploratoires* (survie au niveau image, indice sur les vrais positifs, carte d'occlusion) ne sont pas prévus au protocole v1.1 : à étiqueter comme tels et à intégrer au protocole v1.2.
- Un indice de spécificité élevé en A1 reflète en partie un effet mécanique (effacer la zone d'une détection la supprime) : lire **par rapport au témoin A2** et, pour le soupçon de raccourci, sur la **survie au niveau image**.
- Pour YOLOv11n et RT-DETR-l : remplacer `load_model`, `predict` et `predict_batch`.